In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/FabricDefect"
print(PROJECT_ROOT)

/content/drive/MyDrive/FabricDefect


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import pickle
from PIL import Image
import matplotlib.pyplot as plt


Load Metadata & Extract Features

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/FabricDefect"
metadata = pd.read_csv(f"{PROJECT_ROOT}/data/metadata.csv")

# Use 1000 images for testing
sample_df = metadata.sample(1000, random_state=42)

def extract_simple_features(filepath):
    img = Image.open(filepath).convert("RGB").resize((32,32))
    arr = np.array(img)
    mean_r = np.mean(arr[:,:,0])
    mean_g = np.mean(arr[:,:,1])
    mean_b = np.mean(arr[:,:,2])
    return np.array([mean_r, mean_g, mean_b])

X = np.array([extract_simple_features(f) for f in sample_df['filepath']])
y = sample_df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000, 3)
y shape: (1000,)


In [ ]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

def gini_impurity(y):
    counts = Counter(y)
    total = len(y)
    return 1.0 - sum((count/total)**2 for count in counts.values())


Pruning Logic

In [ ]:
def prune_tree(node, X_val, y_val):
    # If leaf, nothing to prune
    if node.value is not None:
        return node

    # Prune left and right children recursively
    node.left = prune_tree(node.left, X_val, y_val)
    node.right = prune_tree(node.right, X_val, y_val)

    # If both children are leaves, check if we can prune
    if node.left.value is not None and node.right.value is not None:
        # Current predictions with subtree
        y_pred_subtree = np.array([predict_tree(node, x) for x in X_val])
        acc_subtree = np.mean(y_pred_subtree == y_val)

        # Predict with node as leaf (majority class)
        leaf_value = Counter([node.left.value, node.right.value]).most_common(1)[0][0]
        temp_node = Node(value=leaf_value)
        y_pred_leaf = np.array([predict_tree(temp_node, x) for x in X_val])
        acc_leaf = np.mean(y_pred_leaf == y_val)

        # If pruning improves or keeps accuracy, prune
        if acc_leaf >= acc_subtree:
            return temp_node

    return node


Prediction Function

In [ ]:
def predict_tree(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature_index] <= node.threshold:
        return predict_tree(node.left, x)
    else:
        return predict_tree(node.right, x)

def predict(node, X):
    return np.array([predict_tree(node, x) for x in X])


Train Tree on Larger Subset

In [ ]:
def split_dataset(X, y, feature_index, threshold):
    left_idx = np.where(X[:, feature_index] <= threshold)[0]
    right_idx = np.where(X[:, feature_index] > threshold)[0]
    return X[left_idx], X[right_idx], y[left_idx], y[right_idx]

def best_split(X, y):
    n_features = X.shape[1]
    best_gini = 1.0
    best_index = None
    best_threshold = None

    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])
        for threshold in thresholds:
            _, _, y_left, y_right = split_dataset(X, y, feature_index, threshold)
            if len(y_left)==0 or len(y_right)==0:
                continue
            gini_left = gini_impurity(y_left)
            gini_right = gini_impurity(y_right)
            gini_total = (len(y_left)*gini_left + len(y_right)*gini_right)/len(y)

            if gini_total < best_gini:
                best_gini = gini_total
                best_index = feature_index
                best_threshold = threshold
    return best_index, best_threshold

def build_tree(X, y, max_depth=5, min_samples_split=2, depth=0):
    if len(set(y)) == 1:
        return Node(value=y[0])
    if depth >= max_depth or len(y) < min_samples_split:
        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    feature_index, threshold = best_split(X, y)
    if feature_index is None:
        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    X_left, X_right, y_left, y_right = split_dataset(X, y, feature_index, threshold)
    left_child = build_tree(X_left, y_left, max_depth, min_samples_split, depth+1)
    right_child = build_tree(X_right, y_right, max_depth, min_samples_split, depth+1)

    return Node(feature_index, threshold, left_child, right_child)


In [ ]:
tree = build_tree(X, y, max_depth=5)


Prune Tree Using Validation Set

In [ ]:
# Use 200 samples as validation
val_df = metadata.sample(200, random_state=24)
X_val = np.array([extract_simple_features(f) for f in val_df['filepath']])
y_val = val_df['label'].values

# Prune tree
tree = prune_tree(tree, X_val, y_val)
print("Tree pruning completed.")


Tree pruning completed.


Featue Importance

In [ ]:
def feature_importance(node, importance=None):
    if importance is None:
        importance = {}
    if node.value is None:
        importance[node.feature_index] = importance.get(node.feature_index,0)+1
        feature_importance(node.left, importance)
        feature_importance(node.right, importance)
    return importance

importance = feature_importance(tree)
print("Feature importance:", importance)


Feature importance: {0: 7, 2: 3, 1: 4}


Visualize Tree

In [ ]:
def print_tree(node, depth=0):
    if node.value is not None:
        print("\t"*depth + f"Leaf: {node.value}")
        return
    print("\t"*depth + f"[X{node.feature_index} <= {node.threshold}]")
    print_tree(node.left, depth+1)
    print_tree(node.right, depth+1)

print_tree(tree)


[X0 <= 55.353515625]
	[X0 <= 0.5751953125]
		[X0 <= 0.3837890625]
			Leaf: hole
			[X0 <= 0.3857421875]
				Leaf: horizontal
				Leaf: hole
		[X2 <= 86.0419921875]
			Leaf: hole
			[X1 <= 95.66015625]
				Leaf: hole
				[X1 <= 134.37109375]
					Leaf: lines
					Leaf: fabricfaultdetection
	[X0 <= 154.1142578125]
		[X2 <= 47.16796875]
			[X1 <= 7.7001953125]
				Leaf: hole
				Leaf: lines
			[X0 <= 129.0185546875]
				[X1 <= 41.4677734375]
					Leaf: lines
					Leaf: defect free
				Leaf: defect free
		[X2 <= 152.0732421875]
			[X0 <= 158.40625]
				Leaf: defect free
				Leaf: fabricfaultdetection
			Leaf: stain


save pruned tree to drive

In [ ]:
tree_path = f"{PROJECT_ROOT}/utils/decision_tree_pruned.pkl"
with open(tree_path, "wb") as f:
    pickle.dump(tree, f)

print("Pruned tree saved at:", tree_path)


Pruned tree saved at: /content/drive/MyDrive/FabricDefect/utils/decision_tree_pruned.pkl


Implement Conv2D + BatchNorm + ReLU Block

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

def conv_bn_relu(filters, kernel_size=3, strides=1, padding='same'):
    return tf.keras.Sequential([
        layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU()
    ])

# Example usage:
block = conv_bn_relu(64)
x = tf.random.normal((1, 32, 32, 3))
out = block(x)
print("Output shape:", out.shape)  # (1, 32, 32, 64)

Output shape: (1, 32, 32, 64)


Mount Drive + Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Set Dataset Paths

In [ ]:
dataset1_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset"
dataset2_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabricfaultdetection"

clean_data_path = "/content/drive/MyDrive/FabricDefect/clean_data"
os.makedirs(clean_data_path, exist_ok=True)


Understand Folder Structure

In [ ]:
import os
import shutil

# Paths based on your real directory structure
dataset1_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset"
dataset2_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabricfaultdetection"

output_dataset_path = "/content/drive/MyDrive/FabricDefect/dataset"
os.makedirs(output_dataset_path, exist_ok=True)

def merge_defect_folders(src_path, dst_path):
    for defect_type in os.listdir(src_path):
        defect_folder_path = os.path.join(src_path, defect_type)

        if not os.path.isdir(defect_folder_path):
            continue

        # Create output defect folder
        dst_defect_folder = os.path.join(dst_path, defect_type)
        os.makedirs(dst_defect_folder, exist_ok=True)

        # Copy images
        for file_name in os.listdir(defect_folder_path):
            src_file = os.path.join(defect_folder_path, file_name)

            # Ensure it's a file before attempting to copy
            if os.path.isfile(src_file):
                dst_file = os.path.join(dst_defect_folder, file_name)

                # Prevent overwriting by renaming duplicates
                if os.path.exists(dst_file):
                    name, ext = os.path.splitext(file_name)
                    file_name = f"{name}_dup{ext}"
                    dst_file = os.path.join(dst_defect_folder, file_name)

                shutil.copy(src_file, dst_file)
            # else: # If it's a directory, you might want to recurse or ignore, for now we ignore
            #     print(f"Skipping directory: {src_file}")

        print(f"Copied all images from: {defect_folder_path}")

# --- Corrected logic for merging datasets ---

# For dataset1: Adjust the path to the actual directory containing defect type subfolders
actual_dataset1_defect_root = os.path.join(dataset1_path, 'Fabric Defects Dataset', 'Fabric Defect Dataset')

if os.path.exists(actual_dataset1_defect_root) and os.path.isdir(actual_dataset1_defect_root):
    print(f"Merging dataset1 from: {actual_dataset1_defect_root}")
    merge_defect_folders(actual_dataset1_defect_root, output_dataset_path)
else:
    print(f"Warning: Actual dataset1 defect root not found or not a directory: {actual_dataset1_defect_root}")

# For dataset2: It seems to contain images directly under its root.
# We will assume these are all of a certain defect type (e.g., 'fabricfaultdetection' based on its folder name)
# or need a specific label.
print(f"Merging dataset2 from: {dataset2_path}")
default_defect_type_dataset2 = "fabricfaultdetection" # Assuming a default label for dataset2 images
dst_defect_folder_dataset2 = os.path.join(output_dataset_path, default_defect_type_dataset2)
os.makedirs(dst_defect_folder_dataset2, exist_ok=True)

for file_name in os.listdir(dataset2_path):
    src_file = os.path.join(dataset2_path, file_name)
    if os.path.isfile(src_file) and file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        dst_file = os.path.join(dst_defect_folder_dataset2, file_name)

        # Prevent overwriting by renaming duplicates
        if os.path.exists(dst_file):
            name, ext = os.path.splitext(file_name)
            file_name = f"{name}_dup{ext}"
            dst_file = os.path.join(dst_defect_folder_dataset2, file_name)
        shutil.copy(src_file, dst_file)

print("Dataset merge completed!")

Merging dataset1 from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset/Vertical
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset/defect free
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset/hole
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset/horizontal
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset/lines
Copied all images from: /content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset/Fabric Defects Dataset/Fabric Defect Dataset

In [ ]:
dataset1_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabric_defects_dataset"
dataset2_path = "/content/drive/MyDrive/FabricDefect/data/raw/fabricfaultdetection"

print("Dataset 1 folders:", os.listdir(dataset1_path))
print("Dataset 2 folders:", os.listdir(dataset2_path))


Dataset 1 folders: ['fabric-defects-dataset.zip', 'Fabric Defects Dataset']
Dataset 2 folders: ['a00009.jpg', 'a00004.jpg', 'a00005.jpg', 'a00002.jpg', 'a00006.jpg', 'a00008.jpg', 'a00007.jpg', 'a00001.jpg', 'a00027.jpg', 'b100017.png', 'a00060.jpg', 'b100006.png', 'a00018.jpg', 'b100031.png', 'a00070.jpg', 'a00015.jpg', 'b100007.png', 'b100039.png', 'b100037.png', 'b100022.png', 'b100038.png', 'a00074.jpg', 'a00073.jpg', 'b100014.png', 'a00098.jpg', 'a00067.jpg', 'b100011.png', 'b100016.png', 'b100001.png', 'a00010.jpg', 'a00023.jpg', 'a00056.jpg', 'b100030.png', 'b100041.png', 'a00101.jpg', 'b100003.png', 'a00084.jpg', 'a00022.jpg', 'a00075.jpg', 'a00055.jpg', 'b100020.png', 'a00061.jpg', 'b100008.png', 'b100019.png', 'a00026.jpg', 'b100025.png', 'a00100.jpg', 'b100013.png', 'a00083.jpg', 'a00080.jpg', 'a00099.jpg', 'b100042.png', 'a00079.jpg', 'a00090.jpg', 'a00020.jpg', 'b100034.png', 'a00076.jpg', 'b100005.png', 'a00064.jpg', 'a00054.jpg', 'b100023.png', 'a00096.jpg', 'a00058.jpg'

In [ ]:
label_map = {
    "hole": "HOLE",
    "stain": "STAIN",
    "Vertical": "VERTICAL",
    "defect free": "DEFECT_FREE",
    "horizontal": "HORIZONTAL",
    "lines": "LINES",
    "fabricfaultdetection": "FABRIC_FAULT_DETECTION"
}

Function to Clean + Resize Images

In [ ]:
def load_and_clean_image(img_path, size=(224, 224)):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, size)
    return img


Process Dataset 1 + Dataset 2 + Store Cleaned Data

In [ ]:


# Ensure output_dataset_path is defined from previous steps
if 'output_dataset_path' not in locals():
    output_dataset_path = "/content/drive/MyDrive/FabricDefect/dataset"

# Ensure clean_data_path is defined
if 'clean_data_path' not in locals():
    clean_data_path = "/content/drive/MyDrive/FabricDefect/clean_data"
    os.makedirs(clean_data_path, exist_ok=True)

print(f"Processing images from: {output_dataset_path}")
print(f"Saving cleaned images to: {clean_data_path}")

for defect_folder_name in os.listdir(output_dataset_path):
    defect_folder_path = os.path.join(output_dataset_path, defect_folder_name)

    if not os.path.isdir(defect_folder_path):
        print(f"Skipping non-directory item in output_dataset_path: {defect_folder_name}")
        continue

    # Get the label from the map, or use the folder name if not mapped (and convert to uppercase)
    label = label_map.get(defect_folder_name, defect_folder_name.replace(' ', '_').upper())

    save_folder = os.path.join(clean_data_path, label)
    os.makedirs(save_folder, exist_ok=True)

    for img_name in tqdm(os.listdir(defect_folder_path), desc=f"Processing {label}"):
        img_path = os.path.join(defect_folder_path, img_name)
        img = load_and_clean_image(img_path)

        if img is None:
            print(f"Warning: Could not load or clean image: {img_path}")
            continue

        save_path = os.path.join(save_folder, img_name)
        cv2.imwrite(save_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

print("Image cleaning and saving completed.")

Processing images from: /content/drive/MyDrive/FabricDefect/dataset
Saving cleaned images to: /content/drive/MyDrive/FabricDefect/clean_data


Processing FABRIC_DEFECTS_DATASET: 0it [00:00, ?it/s]
Processing FABRIC_FAULT_DETECTION: 100%|██████████| 122/122 [00:02<00:00, 43.23it/s]

Image cleaning and saving completed.


Visualize Random Images from Each Class

In [ ]:
def show_samples(path, n=5):
    for defect in os.listdir(path):
        print(f"Showing samples for: {defect}")
        defect_folder = os.path.join(path, defect)
        imgs = os.listdir(defect_folder)[:n]

        plt.figure(figsize=(15, 3))
        for i, img_name in enumerate(imgs):
            img = cv2.imread(os.path.join(defect_folder, img_name))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            plt.subplot(1, n, i + 1)
            plt.imshow(img)
            plt.axis("off")
        plt.show()

show_samples(clean_data_path, n=5)


Showing samples for: FABRIC_DEFECTS_DATASET


<Figure size 1500x300 with 0 Axes>

Showing samples for: VERTICAL


Showing samples for: DEFECT_FREE


Showing samples for: HOLE


Showing samples for: HORIZONTAL


Showing samples for: LINES


Showing samples for: STAIN


Showing samples for: FABRIC_FAULT_DETECTION


Check Class Imbalance

In [ ]:
import pandas as pd

counts = {}
for defect in os.listdir(clean_data_path):
    folder = os.path.join(clean_data_path, defect)
    counts[defect] = len(os.listdir(folder))

df = pd.DataFrame.from_dict(counts, orient='index', columns=['count'])
df


,count
FABRIC_DEFECTS_DATASET,0
VERTICAL,101
DEFECT_FREE,1666
HOLE,281
HORIZONTAL,136
LINES,157
STAIN,398
FABRIC_FAULT_DETECTION,122


from matplotlib import pyplot as plt
df['count'].plot(kind='hist', bins=20, title='count')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
df['count'].plot(kind='line', figsize=(8, 4), title='count')
plt.gca().spines[['top', 'right']].set_visible(False)

Visualize:

In [ ]:
df.plot(kind="bar", figsize=(10, 4))
plt.title("Class Distribution")
plt.show()


Create Train/Test Split

In [ ]:
import os
import shutil
import numpy as np

def custom_train_test_split(images, test_size=0.2, random_state=42):
    """
    Custom implementation of train_test_split for splitting image lists

    Parameters:
    -----------
    images : list
        List of image filenames
    test_size : float
        Proportion of images for test set
    random_state : int
        Random seed for reproducibility

    Returns:
    --------
    train_images, test_images : tuple of lists
    """
    np.random.seed(random_state)

    # Shuffle the images
    shuffled_images = images.copy()
    np.random.shuffle(shuffled_images)

    # Calculate split point
    n_total = len(shuffled_images)
    n_test = int(n_total * test_size)

    # Split the images
    test_images = shuffled_images[:n_test]
    train_images = shuffled_images[n_test:]

    return train_images, test_images

def split_image_dataset(clean_data_path, test_size=0.2, random_state=42):
    """
    Split image dataset into train and test sets, organizing them in separate directories

    Parameters:
    -----------
    clean_data_path : str
        Path to the directory containing defect folders with images
    test_size : float
        Proportion of images for test set (default: 0.2)
    random_state : int
        Random seed for reproducibility (default: 42)

    Returns:
    --------
    train_path, test_path : tuple of strings
    """
    # Create output directories
    train_path = clean_data_path + "_train"
    test_path = clean_data_path + "_test"

    os.makedirs(train_path, exist_ok=True)
    os.makedirs(test_path, exist_ok=True)

    # Get all defect folders
    defects = [d for d in os.listdir(clean_data_path)
               if os.path.isdir(os.path.join(clean_data_path, d))]

    for defect in defects:
        defect_folder = os.path.join(clean_data_path, defect)

        # Get all images in the defect folder
        images = [img for img in os.listdir(defect_folder)
                  if os.path.isfile(os.path.join(defect_folder, img))]

        # Only proceed with splitting if there are images in the folder
        if images:
            # Use custom train_test_split
            train_imgs, test_imgs = custom_train_test_split(
                images, test_size=test_size, random_state=random_state
            )

            # Create defect subdirectories in train and test paths
            os.makedirs(os.path.join(train_path, defect), exist_ok=True)
            os.makedirs(os.path.join(test_path, defect), exist_ok=True)

            # Copy training images
            for img in train_imgs:
                src_path = os.path.join(defect_folder, img)
                dst_path = os.path.join(train_path, defect, img)
                shutil.copy(src_path, dst_path)

            # Copy test images
            for img in test_imgs:
                src_path = os.path.join(defect_folder, img)
                dst_path = os.path.join(test_path, defect, img)
                shutil.copy(src_path, dst_path)

            print(f"Defect '{defect}': {len(train_imgs)} train, {len(test_imgs)} test images")
        else:
            print(f"Skipping defect '{defect}' as it contains no images.")

    return train_path, test_path

# Alternative stratified version for balanced splits
def stratified_split_image_dataset(clean_data_path, test_size=0.2, random_state=42):
    """
    Split image dataset with stratification (ensuring similar class distribution)
    """
    np.random.seed(random_state)

    train_path = clean_data_path + "_train"
    test_path = clean_data_path + "_test"

    os.makedirs(train_path, exist_ok=True)
    os.makedirs(test_path, exist_ok=True)

    defects = [d for d in os.listdir(clean_data_path)
               if os.path.isdir(os.path.join(clean_data_path, d))]

    for defect in defects:
        defect_folder = os.path.join(clean_data_path, defect)

        # Get all images in the defect folder
        images = [img for img in os.listdir(defect_folder)
                  if os.path.isfile(os.path.join(defect_folder, img))]

        if images:
            # Shuffle images
            np.random.shuffle(images)

            # Calculate test size for this defect
            n_total = len(images)
            n_test = max(1, int(n_total * test_size))  # Ensure at least 1 test image

            # Split images
            test_imgs = images[:n_test]
            train_imgs = images[n_test:]

            # Create directories
            os.makedirs(os.path.join(train_path, defect), exist_ok=True)
            os.makedirs(os.path.join(test_path, defect), exist_ok=True)

            # Copy images
            for img in train_imgs:
                shutil.copy(
                    os.path.join(defect_folder, img),
                    os.path.join(train_path, defect, img)
                )

            for img in test_imgs:
                shutil.copy(
                    os.path.join(defect_folder, img),
                    os.path.join(test_path, defect, img)
                )

            print(f"Defect '{defect}': {len(train_imgs)} train, {len(test_imgs)} test images")
        else:
            print(f"Skipping defect '{defect}' as it contains no images.")

    return train_path, test_path

# Usage example:
if __name__ == "__main__":
    # Example usage (commented out since we don't have actual data)
    # clean_data_path = "path/to/your/clean_data"
    # train_path, test_path = split_image_dataset(clean_data_path, test_size=0.2, random_state=42)

    # Or use the stratified version
    # train_path, test_path = stratified_split_image_dataset(clean_data_path, test_size=0.2, random_state=42)

    pass

Skipping defect 'FABRIC_DEFECTS_DATASET' as it contains no images.


Verify

In [ ]:
import os
import pandas as pd

train_path = clean_data_path + "_train"
test_path = clean_data_path + "_test"

train_counts = {}
test_counts = {}

print("Counting images in train split...")
for defect_type in os.listdir(train_path):
    defect_folder = os.path.join(train_path, defect_type)
    if os.path.isdir(defect_folder):
        train_counts[defect_type] = len(os.listdir(defect_folder))

print("Counting images in test split...")
for defect_type in os.listdir(test_path):
    defect_folder = os.path.join(test_path, defect_type)
    if os.path.isdir(defect_folder):
        test_counts[defect_type] = len(os.listdir(defect_folder))

# Create DataFrames for better visualization
train_df = pd.DataFrame.from_dict(train_counts, orient='index', columns=['train_count'])
test_df = pd.DataFrame.from_dict(test_counts, orient='index', columns=['test_count'])

# Merge and display both counts
split_counts_df = pd.concat([train_df, test_df], axis=1).fillna(0).astype(int)
display(split_counts_df)


Counting images in train split...
Counting images in test split...


,train_count,test_count
VERTICAL,80,21
DEFECT_FREE,1332,334
HOLE,224,57
HORIZONTAL,108,28
LINES,125,32
STAIN,318,80
FABRIC_FAULT_DETECTION,97,25
